#### ANOVA statistical testing 
- ANOVAs ran for data used in main results

In [1]:
# Imports for functionality 
import numpy as np
import pandas as pd
import statsmodels.api as sm

from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from dissertation_simulations.loading_summaries import load_distribution_summary

##### Figure 4 - averaging methods

In [2]:
summary_paths = [
    r"C:\Users\chloe\DissertationSimulations\Result summaries\1D electrodes\exp1bresults1_ts_exp_matrix_radial_-1-1.5.json",
    r"C:\Users\chloe\DissertationSimulations\Result summaries\1D electrodes\exp1bresults2_psd_exp_matrix_radial_-1-1.5.json",
    r"C:\Users\chloe\DissertationSimulations\Result summaries\1D electrodes\exp1bresults3_specmod_exp_matrix_radial_-1-1.5.json"
]

labels = [
    "Average time series",
    "Average linear PSD",
    "Average spectral model"
]

distribution_index = 0 

all_estimates = []
all_methods = []

for path, label in zip(summary_paths, labels):
    electrodes, distributions = load_distribution_summary(path)

    estimates = np.asarray(distributions[distribution_index])

    all_estimates.extend(estimates)
    all_methods.extend([label] * len(estimates))

df = pd.DataFrame({
    "estimate": all_estimates,
    "method": all_methods,
})

df.head()

,estimate,method
0,1.210400,Average time series
1,1.251769,Average time series
2,1.314341,Average time series
3,1.154165,Average time series
4,1.095862,Average time series


In [3]:
# Run the ANOVA - 1 way, effect of averaging method
model = ols("estimate ~ C(method)", data=df).fit()

anova_results = sm.stats.anova_lm(model, typ=2)

print(anova_results)

             sum_sq     df          F    PR(>F)
C(method)  0.037153    2.0  10.874341  0.000028
Residual   0.507355  297.0        NaN       NaN


In [5]:
# Post hoc comparisons 
tukey_results = pairwise_tukeyhsd(
    endog=df["estimate"],
    groups=df["method"],
    alpha=0.05,
)

print(tukey_results)

                Multiple Comparison of Means - Tukey HSD, FWER=0.05                 
        group1                 group2         meandiff p-adj   lower   upper  reject
------------------------------------------------------------------------------------
    Average linear PSD Average spectral model   0.0238 0.0002    0.01  0.0376   True
    Average linear PSD    Average time series   0.0004 0.9971 -0.0133  0.0142  False
Average spectral model    Average time series  -0.0234 0.0002 -0.0372 -0.0096   True
------------------------------------------------------------------------------------


In [ ]:
# omnibus effect sizes: partial eta-squared
residual_ss = two_way_anova.loc["Residual", "sum_sq"]

for factor in ["C(method)", "C(location)", "C(method):C(location)"]:
    partial_eta_squared = (
        two_way_anova.loc[factor, "sum_sq"]
        / (two_way_anova.loc[factor, "sum_sq"] + residual_ss)
    )
    print(f"{factor} partial eta-squared: {partial_eta_squared:.3f}")